In [18]:
RUN_TARGET = "colab"  # "colab" | "local"

## Setup Instructions

### Running on Google Colab
1. Set `RUN_TARGET = "colab"` in Cell 1.
2. Runtime > Change runtime type > T4 GPU or A100.
3. Run the pip-install cell.
4. Run the Drive-mount cell.
5. Run the runtime setup cell to download data, the shared utility modules, and a fresh ESM-2 snapshot.
6. Run the remaining cells top to bottom.
7. Outputs are copied to `Google Drive > My Drive > XAllergen2.0 > models/` and `results/`.

### Running locally on macOS (M-series)
1. Set `RUN_TARGET = "local"` in Cell 1.
2. The Colab setup cells are skipped automatically.
3. MPS is used when available, otherwise CPU.
4. Outputs are saved to the local `models/` and `results/` directories.


In [19]:
if RUN_TARGET == "colab":
    import importlib.metadata as _md
    import subprocess as _sp
    import sys as _sys

    _required = {
        "numpy": "1.26.4",
        "scipy": "1.15.3",
        "scikit-learn": "1.8.0",
        "captum": "0.8.0",
        "transformers": "4.48.1",
        "huggingface-hub": "0.36.2",
    }
    _optional = ["statsmodels"]

    def _version_matches(name: str, expected: str) -> bool:
        try:
            return _md.version(name) == expected
        except _md.PackageNotFoundError:
            return False

    _missing_or_mismatched = [
        f"{name}=={version}"
        for name, version in _required.items()
        if not _version_matches(name, version)
    ]

    for name in _optional:
        try:
            _md.version(name)
        except _md.PackageNotFoundError:
            _missing_or_mismatched.append(name)

    if _missing_or_mismatched:
        print("Installing:", ", ".join(_missing_or_mismatched))
        _sp.run([
            _sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--upgrade",
            *_missing_or_mismatched,
        ], check=True)
        raise SystemExit(
            "Colab environment updated. Restart the runtime once, then rerun the notebook from the top."
        )
    else:
        print("Colab environment already compatible. No reinstall needed.")
else:
    print("Local environment detected. Skipping Colab setup.")


Colab environment already compatible. No reinstall needed.


In [20]:
if RUN_TARGET == "colab":
    from google.colab import drive as _drive
    from pathlib import Path

    _drive.mount("/content/drive", force_remount=False)

    DRIVE_ROOT = Path("/content/drive/MyDrive/XAllergen2.0")
    DRIVE_MODELS = DRIVE_ROOT / "models"
    DRIVE_RESULTS = DRIVE_ROOT / "results"
    DRIVE_MODELS.mkdir(parents=True, exist_ok=True)
    DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)

    print(f"Google Drive mounted: {DRIVE_ROOT}")
    print(f"Models will sync to: {DRIVE_MODELS}")
    print(f"Results will sync to: {DRIVE_RESULTS}")
else:
    print("Local run: skipping Google Drive mount.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted: /content/drive/MyDrive/XAllergen2.0
Models will sync to: /content/drive/MyDrive/XAllergen2.0/models
Results will sync to: /content/drive/MyDrive/XAllergen2.0/results


In [21]:
if RUN_TARGET == "colab":
    import os
    import shutil as _shutil
    import sys
    import urllib.request as _urlreq
    from pathlib import Path

    from huggingface_hub import snapshot_download

    def add_src_to_path(project_root: Path, prepend: bool = True) -> bool:
        src_dir = project_root / "src"
        package_dir = src_dir / "xallergen"
        if package_dir.exists():
            if str(src_dir) in sys.path:
                sys.path.remove(str(src_dir))
            if prepend:
                sys.path.insert(0, str(src_dir))
            else:
                sys.path.append(str(src_dir))
            return True
        return False

    RUNTIME_ROOT = Path("/content/XAllergen2.0")
    _SRC_DIR = RUNTIME_ROOT / "src"
    _PACKAGE_DIR = _SRC_DIR / "xallergen"
    _DATA_DIR = RUNTIME_ROOT / "data"
    _MODEL_DIR = RUNTIME_ROOT / "models"
    _RESULTS_DIR = RUNTIME_ROOT / "results"
    _FRESH_ESM2_DIR = RUNTIME_ROOT / "hf_models" / "facebook_esm2_t6_8M_UR50D"
    for _d in [_DATA_DIR, _MODEL_DIR, _RESULTS_DIR, _FRESH_ESM2_DIR, _PACKAGE_DIR]:
        _d.mkdir(parents=True, exist_ok=True)

    _RAW = "https://raw.githubusercontent.com/Jeffateth/XAllergen2.0/main"
    _copied_from_drive_src = False
    if "DRIVE_ROOT" in globals() and add_src_to_path(DRIVE_ROOT):
        _drive_src_pkg = DRIVE_ROOT / "src" / "xallergen"
        for _module_name in ["__init__.py", "baseline_notebook_utils.py", "mtl_epitope_notebook_utils.py"]:
            _shutil.copy2(_drive_src_pkg / _module_name, _PACKAGE_DIR / _module_name)
            print(f"Copied from Drive src: {_module_name}")
        _copied_from_drive_src = True

    if not _copied_from_drive_src:
        _urlreq.urlretrieve(f"{_RAW}/src/xallergen/__init__.py", _PACKAGE_DIR / "__init__.py")
        for _module_name in ["baseline_notebook_utils.py", "mtl_epitope_notebook_utils.py"]:
            _urlreq.urlretrieve(f"{_RAW}/src/xallergen/{_module_name}", _PACKAGE_DIR / _module_name)
            print(f"Downloaded: {_module_name}")

    for _fname in [
        "deepalgpro_test_cleaned.csv",
        "positives_splitA.csv",
        "positives_splitB.csv",
        "negatives_splitA.csv",
        "negatives_splitB.csv",
    ]:
        _urlreq.urlretrieve(f"{_RAW}/data/{_fname}", _DATA_DIR / _fname)
        print(f"Downloaded: {_fname}")

    _fresh_path = snapshot_download(
        repo_id="facebook/esm2_t6_8M_UR50D",
        local_dir=_FRESH_ESM2_DIR,
        local_dir_use_symlinks=False,
        force_download=True,
        resume_download=False,
    )
    os.environ["XALLERGEN_HF_MODEL_DIR"] = str(_fresh_path)
    print(f"Downloaded fresh ESM-2 snapshot: {_fresh_path}")

    for _checkpoint_name in ["baseline_frozen_esm2.pt", "mtl_frozen_esm2_epitope.pt"]:
        _drive_checkpoint = DRIVE_MODELS / _checkpoint_name
        _runtime_checkpoint = _MODEL_DIR / _checkpoint_name
        if _drive_checkpoint.exists():
            _shutil.copy2(_drive_checkpoint, _runtime_checkpoint)
            print(f"Copied checkpoint from Drive: {_drive_checkpoint}")
        else:
            try:
                _urlreq.urlretrieve(f"{_RAW}/models/{_checkpoint_name}", _runtime_checkpoint)
                print(f"Downloaded checkpoint from GitHub: {_checkpoint_name}")
            except Exception as _exc:
                print(f"Checkpoint unavailable from Drive/GitHub: {_checkpoint_name} ({_exc})")

else:
    print("Local run: skipping GitHub download and model snapshot setup.")


Downloaded: baseline_notebook_utils.py
Downloaded: mtl_epitope_notebook_utils.py
Downloaded: deepalgpro_test_cleaned.csv
Downloaded: positives_splitA.csv
Downloaded: positives_splitB.csv
Downloaded: negatives_splitA.csv
Downloaded: negatives_splitB.csv


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:986: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/775 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/31.4M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/31.4M [00:00<?, ?B/s]

tf_model.h5:   0%|          | 0.00/30.3M [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

Downloaded fresh ESM-2 snapshot: /content/XAllergen2.0/hf_models/facebook_esm2_t6_8M_UR50D
Copied checkpoint from Drive: /content/drive/MyDrive/XAllergen2.0/models/baseline_frozen_esm2.pt
Copied checkpoint from Drive: /content/drive/MyDrive/XAllergen2.0/models/mtl_frozen_esm2_epitope.pt


# 04 - Multi-Task Epitope Supervision for XAllergen2.0

This notebook keeps the experiment flow readable by moving implementation details into `mtl_epitope_notebook_utils.py`.

What the notebook now does:
- configure paths and runtime
- prepare mixed protein-level and residue-level supervision splits
- initialize the MTL model from the notebook 03 baseline checkpoint
- train with early stopping
- evaluate sequence-level and residue-level performance
- save the trained checkpoint and evaluation outputs for centralized probe generation in notebook 06

What lives in the utility module:
- data auditing and split preparation
- dataset and dataloader plumbing
- MTL training and evaluation routines
- probing summaries and plotting helpers


In [22]:
import os
import sys
from pathlib import Path

if RUN_TARGET == "colab":
    RUNTIME_ROOT = Path("/content/XAllergen2.0")
    if str(RUNTIME_ROOT) not in sys.path:
        sys.path.insert(0, str(RUNTIME_ROOT))
    _SRC_DIR = RUNTIME_ROOT / "src"
    if _SRC_DIR.exists() and str(_SRC_DIR) not in sys.path:
        sys.path.insert(0, str(_SRC_DIR))


In [23]:
import sys
from pathlib import Path

# Allow imports from the src-layout package without a root-level shim.
for _candidate in [Path.cwd(), *Path.cwd().parents]:
    _src_dir = _candidate / "src"
    if (_src_dir / "xallergen").exists() and str(_src_dir) not in sys.path:
        sys.path.insert(0, str(_src_dir))
        break
import importlib

import xallergen.baseline_notebook_utils as baseline_notebook_utils
import xallergen.mtl_epitope_notebook_utils as mtl_epitope_notebook_utils

importlib.reload(baseline_notebook_utils)
importlib.reload(mtl_epitope_notebook_utils)

from xallergen.baseline_notebook_utils import (
    DROPOUT,
    ESM_MODEL_NAME,
    HF_MODEL_NAME,
    HIDDEN_DIM,
    IG_STEPS,
    RANDOM_STATE,
    build_tokenizer,
    assert_backbone_trainability_mode,
    configure_backbone_trainability,
    configure_matplotlib_cache,
    detect_device,
    find_project_root,
    initialize_mtl_from_baseline_checkpoint,
    print_runtime_context,
    seed_everything,
)
from xallergen.mtl_epitope_notebook_utils import (
    MTLDataPaths,
    MTLHyperparameters,
    MTLOutputPaths,
    build_dataloaders,
    build_eval_dataloader,
    compute_loss_weights,
    compute_occlusion_scores_mtl,
    evaluate_saved_mtl_checkpoint,
    prepare_classification_benchmark_frame,
    prepare_mtl_splits,
    print_training_balance_summary,
    summarize_split_bundle,
    train_mtl_model,
)

if RUN_TARGET == "colab":
    import matplotlib
    matplotlib.use("Agg")
else:
    configure_matplotlib_cache(Path.cwd())

import json
import matplotlib.pyplot as plt
import pandas as pd
import torch


## Runtime And Hyperparameters

This is the only cell you should usually edit when rerunning the experiment. Paths are derived from the project root, and the training knobs are grouped into one small config object.


In [24]:
BACKBONE_TRAIN_MODE = "frozen"

HPARAMS = MTLHyperparameters(
    classification_batch_size=24,
    epochs=30,
    patience=5,
    learning_rate=1e-3,
    weight_decay=1e-4,
    lambda_cls=1.0,
    lambda_epi=0.5,
    epitope_hidden_dim=128,
    val_fraction=0.1,
    use_protein_pos_weight=False,
    protein_imbalance_tolerance=0.1,
    n_random_draws=100,
    ig_internal_batch_size=1,
)

seed_everything(RANDOM_STATE)

if RUN_TARGET == "colab":
    PROJECT_ROOT = RUNTIME_ROOT
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Device: {DEVICE}")
    if DEVICE == "cuda":
        print(f"  GPU: {torch.cuda.get_device_name(0)}")
        print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    else:
        print("  WARNING: no GPU detected - IG attribution will be slow.")
else:
    PROJECT_ROOT = find_project_root(Path.cwd())
    DEVICE = detect_device()
    print_runtime_context(DEVICE, PROJECT_ROOT)

DATA_DIR = PROJECT_ROOT / "data"
MODEL_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"
CLASSIFICATION_BENCHMARK_CSV = DATA_DIR / "deepalgpro_test_cleaned.csv"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
for _output_subdir in [
    RESULTS_DIR / "classification",
    RESULTS_DIR / "probing" / "rows",
    RESULTS_DIR / "probing" / "summaries",
    RESULTS_DIR / "figures" / "diagnostics",
]:
    _output_subdir.mkdir(parents=True, exist_ok=True)

DATA_PATHS = MTLDataPaths(
    positive_train_csv=DATA_DIR / "positives_splitA.csv",
    positive_test_csv=DATA_DIR / "positives_splitB.csv",
    negative_train_csv=DATA_DIR / "negatives_splitA.csv",
    negative_test_csv=DATA_DIR / "negatives_splitB.csv",
)
_output_paths_kwargs = {
    "baseline_checkpoint_path": MODEL_DIR / "baseline_frozen_esm2.pt",
    "checkpoint_path": MODEL_DIR / "mtl_frozen_esm2_epitope.pt",
    "metrics_path": RESULTS_DIR / "classification" / "mtl_baseline_metrics.json",
    "probe_rows_path": RESULTS_DIR / "probing" / "rows" / "mtl_probing_rows.csv",
    "baseline_probe_rows_path": RESULTS_DIR / "probing" / "rows" / "baseline_probing_rows.csv",
    "combined_probe_rows_path": None,
    "probe_summary_path": RESULTS_DIR / "probing" / "summaries" / "mtl_probing_summary.csv",
    "compare_summary_path": RESULTS_DIR / "probing" / "summaries" / "mtl_vs_baseline_summary.csv",
    "combined_violins_png": RESULTS_DIR / "figures" / "diagnostics" / "mtl_vs_baseline_probing_violins.png",
    "combined_auroc_density_png": RESULTS_DIR / "figures" / "diagnostics" / "mtl_vs_baseline_probing_auroc_vs_density.png",
    "combined_auprc_density_png": RESULTS_DIR / "figures" / "diagnostics" / "mtl_vs_baseline_probing_auprc_vs_density.png",
    "baseline_summary_csv": RESULTS_DIR / "probing" / "summaries" / "probing_summary.csv",
    "mtl_family_label": "MTL (04 frozen)",
    "baseline_family_label": "Baseline (03 frozen)",
}

from dataclasses import fields as _dataclass_fields
_supported_output_fields = {field.name for field in _dataclass_fields(MTLOutputPaths)}
OUTPUT_PATHS = MTLOutputPaths(
    **{key: value for key, value in _output_paths_kwargs.items() if key in _supported_output_fields}
)

tokenizer = build_tokenizer(HF_MODEL_NAME)
ARCHITECTURE_HPARAMS = {
    "hidden_dim": HIDDEN_DIM,
    "dropout": DROPOUT,
    "epitope_hidden_dim": HPARAMS.epitope_hidden_dim,
}


Device: cuda
  GPU: Tesla T4
  VRAM: 15.6 GB


## Data Preparation

The utility module handles CSV auditing, epitope-label parsing, sequence length filtering, and the train/validation/test assembly. The notebook keeps only the high-level split summary.


In [25]:
split_bundle = prepare_mtl_splits(DATA_PATHS, val_fraction=HPARAMS.val_fraction)
summarize_split_bundle(split_bundle)

train_loader, val_loader, test_loader = build_dataloaders(
    split_bundle["train_mixed_df"],
    split_bundle["val_mixed_df"],
    split_bundle["test_mixed_df"],
    tokenizer=tokenizer,
    batch_size=HPARAMS.classification_batch_size,
)

weight_info = compute_loss_weights(
    split_bundle["positive_train_df"],
    split_bundle["negative_train_df"],
    device=DEVICE,
    use_protein_pos_weight=HPARAMS.use_protein_pos_weight,
    protein_imbalance_tolerance=HPARAMS.protein_imbalance_tolerance,
)

classification_benchmark_df, classification_benchmark_audit = prepare_classification_benchmark_frame(
    CLASSIFICATION_BENCHMARK_CSV,
    split_name="deepalgpro_test",
    frame_name="classification_benchmark_test",
)
classification_benchmark_loader = build_eval_dataloader(
    classification_benchmark_df,
    tokenizer=tokenizer,
    batch_size=HPARAMS.classification_batch_size,
)
print(
    f"Classification benchmark: {classification_benchmark_audit['kept_rows']} sequences from {CLASSIFICATION_BENCHMARK_CSV.name}"
)


Data audit:
  positive_train_full: raw_rows=313, kept_rows=313, dropped_over_max_len=0
  positive_test: raw_rows=61, kept_rows=61, dropped_over_max_len=0
  negative_train_full: raw_rows=299, kept_rows=299, dropped_over_max_len=0
  negative_test: raw_rows=75, kept_rows=75, dropped_over_max_len=0
Post-filter split inputs: positive_train_full=313 positive_test=61 negative_train_full=299 negative_test=75
Mixed train/val/test: 550 62 136
Positive train/val/test: 281 32 61
Negative train/val/test: 269 30 75
Positive train density mean: 0.235
Positive test density mean: 0.2499
Classification benchmark: 1377 sequences from deepalgpro_test_cleaned.csv


## Model Initialization

We start from the notebook 03 checkpoint so the backbone, attention pooling, and protein classifier head are reused. Only the new epitope head is freshly initialized.


In [9]:
if not OUTPUT_PATHS.baseline_checkpoint_path.exists():
    raise FileNotFoundError(
        f"Baseline checkpoint not found: {OUTPUT_PATHS.baseline_checkpoint_path}. "
        "Run notebook 03 first or copy baseline_frozen_esm2.pt into models/."
    )

model, baseline_checkpoint = initialize_mtl_from_baseline_checkpoint(
    OUTPUT_PATHS.baseline_checkpoint_path,
    DEVICE,
    model_name=HF_MODEL_NAME,
    hidden_dim=HIDDEN_DIM,
    dropout=DROPOUT,
    epitope_hidden_dim=HPARAMS.epitope_hidden_dim,
)

backbone_trainability = configure_backbone_trainability(model, BACKBONE_TRAIN_MODE)
backbone_assertions = assert_backbone_trainability_mode(model, BACKBONE_TRAIN_MODE)
trainable_params = [param for param in model.parameters() if param.requires_grad]
optimizer = torch.optim.AdamW(
    trainable_params,
    lr=HPARAMS.learning_rate,
    weight_decay=HPARAMS.weight_decay,
)

print(f"Backbone final block used for assertions: {backbone_assertions['final_block_path']}")
print_training_balance_summary(
    split_bundle["positive_train_df"],
    split_bundle["negative_train_df"],
    weight_info,
    model,
    trainable_params,
    lambda_cls=HPARAMS.lambda_cls,
    lambda_epi=HPARAMS.lambda_epi,
)


Some weights of EsmModel were not initialized from the model checkpoint at /content/XAllergen2.0/hf_models/facebook_esm2_t6_8M_UR50D and are newly initialized: ['esm.pooler.dense.bias', 'esm.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loaded baseline checkpoint: /content/XAllergen2.0/models/baseline_frozen_esm2.pt
Loaded shared keys:
  attention_pool.score.bias
  attention_pool.score.weight
  backbone.contact_head.regression.bias
  backbone.contact_head.regression.weight
  backbone.embeddings.word_embeddings.weight
  backbone.encoder.emb_layer_norm_after.bias
  backbone.encoder.emb_layer_norm_after.weight
  backbone.encoder.layer.0.LayerNorm.bias
  backbone.encoder.layer.0.LayerNorm.weight
  backbone.encoder.layer.0.attention.LayerNorm.bias
  backbone.encoder.layer.0.attention.LayerNorm.weight
  backbone.encoder.layer.0.attention.output.dense.bias
  backbone.encoder.layer.0.attention.output.dense.weight
  backbone.encoder.layer.0.attention.self.key.bias
  backbone.encoder.layer.0.attention.self.key.weight
  backbone.encoder.layer.0.attention.self.query.bias
  backbone.encoder.layer.0.attention.self.query.weight
  backbone.encoder.layer.0.attention.self.rotary_embeddings.inv_freq
  backbone.encoder.layer.0.attention.

## Training

The full epoch loop, checkpointing, and early stopping live in `mtl_epitope_notebook_utils.py`. This cell is now just the experiment call.


In [10]:
training_run = train_mtl_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    device=DEVICE,
    protein_pos_weight=weight_info["protein_pos_weight"],
    residue_pos_weight=weight_info["residue_pos_weight"],
    lambda_cls=HPARAMS.lambda_cls,
    lambda_epi=HPARAMS.lambda_epi,
    epochs=HPARAMS.epochs,
    patience=HPARAMS.patience,
    trainable_params=trainable_params,
    checkpoint_path=OUTPUT_PATHS.checkpoint_path,
    baseline_checkpoint_path=OUTPUT_PATHS.baseline_checkpoint_path,
    architecture_hyperparameters=ARCHITECTURE_HPARAMS,
)

history_df = pd.DataFrame(training_run["history"])
history_df.tail()


Training:   0%|          | 0/30 [00:00<?, ?epoch/s]

Epoch   1/30 | train_total=0.95430 | train_cls=0.40334 | train_epi=1.09759 | train_lambda_cls=0.40334 | train_lambda_epi=0.54879 | val_total=0.86272 | val_cls=0.32155 | val_epi=1.08133 | val_lambda_cls=0.32155 | val_lambda_epi=0.54066 | best=1
Epoch   2/30 | train_total=0.87089 | train_cls=0.32927 | train_epi=1.07006 | train_lambda_cls=0.32927 | train_lambda_epi=0.53503 | val_total=0.84774 | val_cls=0.31645 | val_epi=1.07584 | val_lambda_cls=0.31645 | val_lambda_epi=0.53792 | best=2
Epoch   3/30 | train_total=0.82692 | train_cls=0.29863 | train_epi=1.04742 | train_lambda_cls=0.29863 | train_lambda_epi=0.52371 | val_total=0.83383 | val_cls=0.29432 | val_epi=1.07967 | val_lambda_cls=0.29432 | val_lambda_epi=0.53983 | best=3
Epoch   4/30 | train_total=0.80319 | train_cls=0.28302 | train_epi=1.03563 | train_lambda_cls=0.28302 | train_lambda_epi=0.51781 | val_total=0.83681 | val_cls=0.29162 | val_epi=1.08489 | val_lambda_cls=0.29162 | val_lambda_epi=0.54244 | best=3
Epoch   5/30 | train_tot

,epoch,train_total_loss,train_cls_loss,train_epi_loss,train_weighted_cls,train_weighted_epi,val_total_loss,val_cls_loss,val_epi_loss,val_weighted_cls,val_weighted_epi
3,4,0.803187,0.283015,1.035627,0.283015,0.517813,0.836809,0.291620,1.084890,0.291620,0.542445
4,5,0.760243,0.250132,1.022823,0.250132,0.511411,0.844486,0.296164,1.094837,0.296164,0.547419
5,6,0.751914,0.243598,1.016824,0.243598,0.508412,0.849837,0.304595,1.096886,0.304595,0.548443
6,7,0.738243,0.229188,1.014640,0.229188,0.507320,0.857760,0.304779,1.114105,0.304779,0.557052
7,8,0.719918,0.215401,1.004893,0.215401,0.502446,0.854246,0.290469,1.124695,0.290469,0.562347


## Evaluation

This stage reloads the best saved checkpoint, computes validation and test metrics, and writes a compact JSON artifact for later comparison.


In [26]:
evaluation = evaluate_saved_mtl_checkpoint(
    checkpoint_path=OUTPUT_PATHS.checkpoint_path,
    device=DEVICE,
    val_loader=val_loader,
    test_loader=test_loader,
    protein_pos_weight=weight_info["protein_pos_weight"],
    residue_pos_weight=weight_info["residue_pos_weight"],
    lambda_cls=HPARAMS.lambda_cls,
    lambda_epi=HPARAMS.lambda_epi,
    baseline_checkpoint_path=OUTPUT_PATHS.baseline_checkpoint_path,
    metrics_path=OUTPUT_PATHS.metrics_path,
    architecture_hyperparameters=ARCHITECTURE_HPARAMS,
    training_hparams=HPARAMS,
    weight_info=weight_info,
    hidden_dim=HIDDEN_DIM,
    dropout=DROPOUT,
    classification_test_loader=classification_benchmark_loader,
    classification_test_name="deepalgpro_test_cleaned",
)

model = evaluation["model"]
history_df = evaluation["history_df"]
print(json.dumps(evaluation["metrics_payload"]["test_metrics"], indent=2))


Some weights of EsmModel were not initialized from the model checkpoint at /content/XAllergen2.0/hf_models/facebook_esm2_t6_8M_UR50D and are newly initialized: ['esm.pooler.dense.bias', 'esm.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Validation classification metrics:
{
  "threshold": 0.5,
  "accuracy": 0.8548387096774194,
  "auroc": 0.95,
  "auprc": 0.9594759048524173,
  "f1": 0.8656716417910447,
  "precision": 0.8285714285714286,
  "recall": 0.90625,
  "specificity": 0.8,
  "mcc": 0.7118330908340446,
  "confusion_matrix": {
    "tn": 24,
    "fp": 6,
    "fn": 3,
    "tp": 29
  }
}
Validation residue metrics:
{
  "n_valid_residues": 9570,
  "n_positive_residues": 1855,
  "auroc": 0.5963394374734694,
  "auprc": 0.2621775707401943,
  "precision_at_k": 0.28463611006736755
}
Test classification metrics:
{
  "threshold": 0.5,
  "accuracy": 0.9099491648511256,
  "auroc": 0.9616514690982776,
  "auprc": 0.9610144590003583,
  "f1": 0.91027496382055,
  "precision": 0.8859154929577465,
  "recall": 0.9360119047619048,
  "specificity": 0.8851063829787233,
  "mcc": 0.821282990772336,
  "confusion_matrix": {
    "tn": 624,
    "fp": 81,
    "fn": 43,
    "tp": 629
  },
  "test_total_loss": 0.2621004201214889,
  "test_cls_loss":

## Centralized Probing

Probe generation now lives in `06_generate_probe_rows.ipynb`. This training notebook stops after checkpointing and evaluation so expensive attribution runs can be batched centrally across all model families.


In [12]:
print("Run 06_generate_probe_rows.ipynb after training to generate frozen-MTL probe artifacts.")


Run 06_generate_probe_rows.ipynb after training to generate frozen-MTL probe artifacts.


## Visualization

Cross-model probe figures are rendered later from centrally generated probe-row CSVs.


In [13]:
print("Probe visualizations are centralized in 07_compare_all_model_probes.ipynb.")


Probe visualizations are centralized in 07_compare_all_model_probes.ipynb.


In [27]:
if RUN_TARGET == "colab":
    import shutil as _shutil

    DRIVE_MODELS.mkdir(parents=True, exist_ok=True)
    DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)

    for _src, _dst_dir in [
        (OUTPUT_PATHS.checkpoint_path, DRIVE_MODELS),
        (OUTPUT_PATHS.metrics_path, DRIVE_RESULTS),
    ]:
        if _src.exists():
            _shutil.copy2(_src, _dst_dir / _src.name)
            print(f"Copied to Drive: {_dst_dir / _src.name}")
        else:
            print(f"Skipped missing output: {_src}")
else:
    print("Local run: outputs saved to:")
    for _out_path in [
        OUTPUT_PATHS.checkpoint_path,
        OUTPUT_PATHS.metrics_path,
    ]:
        print(f"  {_out_path}")


Copied to Drive: /content/drive/MyDrive/XAllergen2.0/models/mtl_frozen_esm2_epitope.pt
Copied to Drive: /content/drive/MyDrive/XAllergen2.0/results/mtl_baseline_metrics.json


## Interpretation Guide

What to inspect after a run:
- `mtl_epitope_notebook_utils.py` for the implementation details behind training, evaluation, and centralized probing helpers.
- `results/mtl_baseline_metrics.json` for sequence-level and flattened residue-level metrics.
- `history_df` for whether validation gains are coming from protein classification, residue supervision, or both.
- `06_generate_probe_rows.ipynb` to generate probe artifacts for this checkpoint alongside the other active model families.
- `07_compare_all_model_probes.ipynb` or `replot_probe_figures.py` for centralized cross-model summaries and figures rendered from saved probe rows.

A healthy outcome here is: protein-level classification stays stable while residue-head probing and attribution alignment improve.
